# 01 — IEEE-CIS Exploratory Data Analysis

**Status:** measured metrics are produced by `python -m src.data.eda` (canonical, no plots).
This notebook adds figures and has not yet been executed (matplotlib not installed).

**Data licence:** IEEE-CIS is distributed under Kaggle competition rules. Do not commit raw
rows. Clear outputs that show individual records before committing; aggregate tables/plots are fine.

Prerequisites:
1. CSVs in `data/raw/ieee-cis/` (see `data/README.md`).
2. `pip install -r requirements-eda.txt` inside `.venv`.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.data.load import HAS_IDENTITY, LABEL, load_ieee_cis

REPORTS = ROOT / "reports"
FIGURES = REPORTS / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
metrics: dict = {}

## 1. Load and overview

In [ ]:
df = load_ieee_cis()  # validates schema, unique keys; left-joins identity
metrics["rows"] = len(df)
metrics["columns"] = df.shape[1]
metrics["memory_mb"] = round(df.memory_usage(deep=True).sum() / 1e6, 1)
metrics["has_identity_rate"] = round(float(df[HAS_IDENTITY].mean()), 4)
print(metrics)
df.dtypes.value_counts()

## 2. Class imbalance

Documents the positive rate and the negative:positive ratio, which is the starting point
for XGBoost `scale_pos_weight`. Accuracy is not a useful metric at this imbalance; we will
use PR-AUC plus precision/recall at an operating threshold.

In [ ]:
counts = df[LABEL].value_counts().sort_index()
metrics["fraud_count"] = int(counts.get(1, 0))
metrics["legit_count"] = int(counts.get(0, 0))
metrics["fraud_rate"] = round(float(df[LABEL].mean()), 5)
metrics["neg_pos_ratio"] = round(metrics["legit_count"] / max(metrics["fraud_count"], 1), 2)
print({k: metrics[k] for k in ["fraud_count", "legit_count", "fraud_rate", "neg_pos_ratio"]})

ax = counts.rename({0: "legit", 1: "fraud"}).plot.bar(logy=True, title="Class counts (log scale)")
ax.set_ylabel("transactions")
plt.tight_layout(); plt.savefig(FIGURES / "class_balance.png", dpi=120); plt.show()

## 3. Fraud over time

`TransactionDT` is seconds from an undisclosed reference point, not an absolute timestamp.
We bucket by day offset. Drift in volume or fraud rate here motivates a **time-based**
train/validation/test split rather than a random split.

In [ ]:
day = (df["TransactionDT"] // 86400).astype(int).rename("day")
daily = df.groupby(day)[LABEL].agg(transactions="size", fraud_rate="mean")
metrics["days_covered"] = int(day.max() - day.min() + 1)
metrics["daily_fraud_rate_min"] = round(float(daily["fraud_rate"].min()), 5)
metrics["daily_fraud_rate_max"] = round(float(daily["fraud_rate"].max()), 5)

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
daily["transactions"].plot(ax=axes[0], title="Daily transactions")
daily["fraud_rate"].rolling(7, min_periods=1).mean().plot(ax=axes[1], title="Fraud rate (7-day mean)")
axes[1].set_xlabel("day offset")
plt.tight_layout(); plt.savefig(FIGURES / "fraud_over_time.png", dpi=120); plt.show()

## 4. Missingness

In [ ]:
missing = df.isna().mean().sort_values(ascending=False)
metrics["columns_over_50pct_missing"] = int((missing > 0.5).sum())
metrics["columns_over_90pct_missing"] = int((missing > 0.9).sum())
print(metrics["columns_over_50pct_missing"], metrics["columns_over_90pct_missing"])

groups = missing.groupby(lambda c: c.rstrip("0123456789_") or c).mean().sort_values()
groups.plot.barh(figsize=(6, 8), title="Mean missing fraction by column family")
plt.tight_layout(); plt.savefig(FIGURES / "missingness.png", dpi=120); plt.show()

# Is missingness itself predictive? (informative-missingness check on key columns)
key_cols = ["card2", "addr1", "dist1", "P_emaildomain", "R_emaildomain", "DeviceInfo", "D1"]
df[key_cols].isna().groupby(df[LABEL]).mean().T.rename(columns={0: "legit", 1: "fraud"})

## 5. Feature availability

Coverage of the source columns needed by each planned feature. The user identifier is a
**proxy** (IEEE-CIS has no true account ID); it will be refined in Milestone 2.

In [ ]:
planned = {
    "transaction_velocity": ["TransactionDT", "card1"],
    "amount_vs_user_average": ["TransactionAmt", "card1", "addr1"],
    "device_change": ["DeviceInfo", "DeviceType"],
    "location_change": ["addr1", "addr2", "dist1"],
    "email_domain_risk": ["P_emaildomain", "R_emaildomain"],
    "graph_entities": ["card1", "addr1", "P_emaildomain", "DeviceInfo"],
}
availability = pd.DataFrame(
    [
        {
            "feature": name,
            "source_columns": ", ".join(cols),
            "all_present_rate": round(float(df[cols].notna().all(axis=1).mean()), 4),
            "any_present_rate": round(float(df[cols].notna().any(axis=1).mean()), 4),
        }
        for name, cols in planned.items()
    ]
)
metrics["feature_availability"] = availability.set_index("feature")[
    ["all_present_rate", "any_present_rate"]
].to_dict(orient="index")
availability

## 6. Transaction amount by class

In [ ]:
amount_stats = df.groupby(LABEL)["TransactionAmt"].describe(percentiles=[0.5, 0.95, 0.99])
metrics["amount_median_by_class"] = amount_stats["50%"].round(2).to_dict()
ax = df.boxplot(column="TransactionAmt", by=LABEL, showfliers=False)
ax.set_yscale("log"); ax.set_title("TransactionAmt by isFraud (no outliers)"); plt.suptitle("")
plt.tight_layout(); plt.savefig(FIGURES / "amount_by_class.png", dpi=120); plt.show()
amount_stats

## 7. Canonical metrics

`reports/eda_metrics.json` is written by `python -m src.data.eda`, not by this notebook,
so the two cannot diverge. The cell below recomputes the same aggregates for display.

In [ ]:
from src.data.eda import compute_eda_metrics

canonical = compute_eda_metrics(df)
json.loads(json.dumps(canonical))  # aggregates only; not written to disk here